# Jämförelse mellan City Hotel och Resort Hotel

## Syfte

Syftet är att undersöka om våra modeller presterar olika för City Hotel och Resort Hotel när de används för att identifiera bokningar med hög risk för avbokning.

Vi jämför vår regelbaserade baseline, Logistic Regression och Random Forest.

## Metod

Modellerna utvärderas på samma valideringsdata.

För varje modell flaggas de 20 % av bokningarna som har högst risk i hela valideringsdatan. Därefter delas resultaten upp efter hotelltyp och precision och recall beräknas separat för City Hotel och Resort Hotel.

Vi använder samma globala gräns för respektive modell, i stället för att flagga 20 % inom varje hotell. Det innebär att andelen flaggade bokningar kan skilja sig mellan hotellen.

Baselinen använder `lead_time` som riskmått, medan Logistic Regression och Random Forest använder modellernas predikterade sannolikheter.

De redan tränade ML-modellerna ska återanvändas utan ny träning.

### Ändra arbetskatalog

In [7]:
import os

# Byt arbetskatalog från notebooks till projektroten om det behövs
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

### Importera paket och egna moduler

In [8]:
from src.data_split import load_cleaned_data, get_splits
from src.evaluation import flag_top_share, evaluate_model

### Ladda data och kronologiska splits

In [9]:
# Läs in städad data från data/processed/
df = load_cleaned_data()

# Hämta de kronologiska uppdelningarna
splits = get_splits(df)
val = splits["val"]

# Kontrollera antal bokningar per hotell
print("Valideringsdata:", val.shape)
print("\nBokningar per hotell:")
print(val["hotel"].value_counts())

2026-10-17 18:21:12 [INFO] src.data_split: Läser in städad data från data\processed\hotel_bookings_cleaned.csv
2026-10-17 18:21:12 [INFO] src.data_split: Split vid 2016-07-01: 40414 träning (avbokade 0.308), 11643 utvärdering (avbokade 0.353)
2026-10-17 18:21:13 [INFO] src.data_split: Split vid 2016-10-01: 55393 träning (avbokade 0.321), 14879 utvärdering (avbokade 0.396)
Valideringsdata: (11643, 34)

Bokningar per hotell:
hotel
City Hotel      7635
Resort Hotel    4008
Name: count, dtype: int64


## Baseline

Baselinen använder `lead_time` som riskmått. Vi undersöker hur de globalt flaggade bokningarna fördelas mellan City Hotel och Resort Hotel.

In [13]:
# Baseline använder lead_time som riskpoäng
scores_baseline = val["lead_time"]

# Flagga de 20 % av alla bokningar som har längst lead_time
val = val.copy()
val["baseline_flag"] = flag_top_share(scores_baseline, share=0.20)

# Kontrollera antal flaggade bokningar
print("Totalt antal flaggade:", val["baseline_flag"].sum())

# Visa antal och andel flaggade per hotell
summary = val.groupby("hotel")["baseline_flag"].agg(
    antal_flaggade="sum",
    antal_bokningar="count"
)

summary["andel_flaggade_%"] = (
    summary["antal_flaggade"] / summary["antal_bokningar"] * 100
)

print(summary.round(1))

Totalt antal flaggade: 2329
              antal_flaggade  antal_bokningar  andel_flaggade_%
hotel                                                          
City Hotel              1085             7635              14.2
Resort Hotel            1244             4008              31.0


### Utvärdering per hotell

Vi jämför de flaggade bokningarna med de faktiska avbokningarna och beräknar precision och recall separat för City Hotel och Resort Hotel.

In [11]:
for hotel in ["City Hotel", "Resort Hotel"]:

    # Välj bokningarna för aktuellt hotell
    hotel_data = val[val["hotel"] == hotel]

    # Beräkna precision och recall
    metrics = evaluate_model(
        hotel_data["is_canceled"],
        hotel_data["baseline_flag"]
    )

    print(f"\n{hotel}")
    print(f"Precision: {metrics['Precision']:.2%}")
    print(f"Recall:    {metrics['Recall']:.2%}")


City Hotel
Precision: 61.84%
Recall:    23.90%

Resort Hotel
Precision: 47.83%
Recall:    45.80%


### Observation – Baseline

Baselinen har högre precision för City Hotel (61,84 %), medan recall är högre för Resort Hotel (45,80 %).

En större andel av bokningarna flaggas på Resort Hotel (31,0 %) än på City Hotel (14,2 %). Det innebär att baselinen hittar en större andel av avbokningarna på Resort Hotel, men är mer träffsäker bland de flaggade bokningarna på City Hotel.

Skillnaderna kan bero på hotellens olika fördelning av `lead_time` och avbokningsmönster, men orsaken kan inte fastställas utifrån denna jämförelse.